### Imports

In [ ]:
import torch, transformers, datasets, json, pickle
import pandas as pd
import numpy as np

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from datasets import Dataset

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt

Checking for CUDA

In [4]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: False


### Setting up Tokenizer

Loaded the label encoder from baseline to keep label space consistent, initialized distilbert-base-uncased tokenizer, and converted train/val parquets to HF Datasets with batched tokenization (max_length=512, truncation + padding) on clean_complaint_text.

In [ ]:
model_name = "distilbert-base-uncased"
train_df = pd.read_parquet("../data/train.parquet")
val_df = pd.read_parquet("../data/val.parquet")

with open("../models/label_encoder.pkl", "rb") as f:
    label_encoder = pickle.load(f)
    
train_df["label"] = label_encoder.transform(train_df["Product"])
val_df["label"] = label_encoder.transform(val_df["Product"])

tokenizer = AutoTokenizer.from_pretrained(model_name)
num_labels = train_df["Product"].nunique()

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

max_length = 512

def tokenize_function(examples):
    return tokenizer(
        examples["clean_complaint_text"],
        truncation=True,
        padding="max_length",
        max_length=max_length
    )

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

train_dataset = train_dataset.map(
    tokenize_function,
    batched=True
)

val_dataset = val_dataset.map(
    tokenize_function,
    batched=True
)

### Setup Training Arguments and Trainer
Configured TrainingArguments for 3 epochs, batch size 16, lr 2e-5 with eval per epoch, save the best of all and load_best_model_at_end=True. Initialized HF Trainer with DistilBERT model, tokenized train/val datasets, and set report_to="none" to handle MLflow logging manually like the baseline.

In [ ]:
training_args = TrainingArguments(
    output_dir="../models/distilbert",

    num_train_epochs=3,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    learning_rate=2e-5,

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss", # or f1/accuracy
    greater_is_better=False, # True if using f1/accuracy
    save_total_limit=1, # keeps only best checkpoint to save disk

    logging_strategy="epoch",

    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

### Training Command

In [ ]:
trainer.train()

### Evaluation

In [ ]:
predictions = trainer.predict(val_dataset)

y_pred = np.argmax(predictions.predictions, axis=1)

y_true = np.array(val_dataset["label"])

accuracy = accuracy_score(y_true, y_pred)

macro_f1 = f1_score(
    y_true,
    y_pred,
    average="macro"
)

weighted_f1 = f1_score(
    y_true,
    y_pred,
    average="weighted"
)

print("Accuracy:", accuracy)
print("Macro F1:", macro_f1)
print("Weighted F1:", weighted_f1)

Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)

labels = label_encoder.classes_

plt.figure(figsize=(10, 8))

plt.imshow(cm)

plt.xticks(range(len(labels)), labels, rotation=45)

plt.yticks(range(len(labels)), labels)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("DistilBERT Confusion Matrix")

plt.colorbar()
plt.tight_layout()
plt.savefig("distilbert_confusion_matrix.png")
plt.show()

print(classification_report(y_true, y_pred, target_names=label_encoder.classes_))

### Saving Metrics and Params for Logging

In [ ]:
metrics = {
    "val_accuracy": accuracy,
    "val_macro_f1": macro_f1,
    "val_weighted_f1": weighted_f1
}

params = {
    "model": "distilbert-base-uncased",
    "max_length": 512,
    "epochs": 3,
    "train_batch_size": 16,
    "eval_batch_size": 16,
    "learning_rate": 2e-5,
    "train_samples": len(train_dataset),
    "val_samples": len(val_dataset),
    "num_labels": num_labels
}

Saved the best fine-tuned checkpoint (loaded via `load_best_model_at_end`) and tokenizer to `../models/distilbert` using `save_pretrained()` for inference and deployment.

In [ ]:
model_path = "../models/distilbert"

trainer.model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)

### MLFlow Logging

In [ ]:
import mlflow
import mlflow.transformers

# -----------------------------
# Create MLflow run
# -----------------------------

with mlflow.start_run(run_name="DistilBERT"):

    # Parameters
    mlflow.log_params(
        {
            "model": params["model"],
            "max_length": params["max_length"],
            "epochs": params["epochs"],
            "train_batch_size": params["train_batch_size"],
            "eval_batch_size": params["eval_batch_size"],
            "learning_rate": params["learning_rate"],
            "train_samples": params["train_samples"],
            "val_samples": params["val_samples"],
            "num_labels": params["num_labels"],
        }
    )

    # Metrics
    mlflow.log_metric("accuracy", metrics["val_accuracy"])

    mlflow.log_metric("macro_f1", metrics["val_macro_f1"])

    mlflow.log_metric("weighted_f1", metrics["val_weighted_f1"])

    # Confusion matrix
    mlflow.log_artifact("distilbert_confusion_matrix.png")

    # Label encoder
    mlflow.log_artifact("../models/label_encoder.pkl", artifact_path="label_encoder")

    # Log DistilBERT as an MLflow Model
    mlflow.transformers.log_model(
        transformers_model={"model": model, "tokenizer": tokenizer},
        name="distilbert_model",
        task="text-classification"
    )

    print("DistilBERT logged successfully.")